# Demo: Using Pandera for Data Quality Checks
This notebook goes through simple examples demonstrating how we can use the open-source pandera package to help with data quality checks. Pandera's schema validation templates can be reused across a data pipeline, making it more maintainable than custom checks. The checks closely follow those in basic_validation_checks.ipynb.

In [1]:
# import packages:

from platform import python_version
import numpy as np
import pandas as pd
import duckdb
from pandera import __version__ as pandera_version
import pandera.pandas as pa
import json # built-in library
from datetime import datetime, timedelta # built-in libraries
from pretty_html_validation_report import generate_dq_report

# verify that package versions are as expected:

print('Python version:', python_version())
print('Numpy version:', np.__version__)
print('Pandas version:', np.__version__)
print('Duckdb version:', duckdb.__version__)
print('Pandera version', pandera_version)


pd.set_option('display.max_columns', None) # don't hide columns in display

Python version: 3.13.12
Numpy version: 2.4.2
Pandas version: 2.4.2
Duckdb version: 1.4.3
Pandera version 0.30.1


In [2]:
# specify input files here:

CONFIG_FILE_PATH = 'config_comp_2020.json'
EXPECTED_VERSION = 1 # this version of the data pipeline requires the config file to be version 1
CLAIMS_FILE_PATH = 'Datasets/claim_details.parquet'
POLICY_FILE_PATH = 'Datasets/policy_details.parquet'

VALIDATION_REPORT_PATH = 'pandera_report.html'


In [3]:
# load in config file first and check that the version matches what is expected. These will be used as hardcoded values in the remainder of the pipeline.
# for future runs, we'll be able to edit just the config file without having to touch the code
with open(CONFIG_FILE_PATH, 'r') as f:
    settings = json.load(f)

print(settings)

assert settings['version'] == EXPECTED_VERSION, 'Version of config file is invalid. Expected: {EXPECTED_VERSION}. Actual: {settings["version"]}'

{'version': 1, 'valuation_date': '2021-01-01', 'earliest_accident_year': 2015, 'latest_accident_year': 2020, 'subline': 'COMP'}


In [5]:
# let's first get the claims table and filter it appropriately:
# (Note: we will pretend that date of intimation represents valuation date and intimation amount is paid loss)

claims = duckdb.sql(f'''
    SELECT
        *
    FROM
        read_parquet('{CLAIMS_FILE_PATH}')
    WHERE
        policytype = '{settings["subline"]}'
        AND YEAR(date_of_accident) >= {settings["earliest_accident_year"]}
        AND YEAR(date_of_accident) <= {settings["latest_accident_year"]}
        AND valuation_date < DATE '{settings["valuation_date"]}'
    ;
''').df()


In [6]:
latest_acc_month = str(settings['latest_accident_year']) + '-12' # last month of the year
latest_val_month = (datetime.strptime(settings['valuation_date'], '%Y-%m-%d') - timedelta(days=1)).strftime('%Y-%m') # get the previous month

claims_schema = pa.DataFrameSchema(
    {
        'CLAIM_NO': pa.Column(str, unique=True, nullable=False),
        'AGE': pa.Column(float, checks=[pa.Check.ge(0), pa.Check.le(150)], nullable=True),
        'VALUATION_DATE': pa.Column(pa.DateTime, nullable=False),
        'DATE_OF_ACCIDENT': pa.Column(pa.DateTime, nullable=False),
        'POLICY_START': pa.Column(pa.DateTime, nullable=False),
        'POLICY_END': pa.Column(pa.DateTime, nullable=False),
        'PAID_AMOUNT': pa.Column(float, checks=[
            pa.Check.ge(0),
            pa.Check.le(1_000_000),
            pa.Check(
                lambda s: ~((s.isna()) | (s.round(0) == s.round(2))).all(),
                name='check_if_all_payments_are_rounded',
                error='All paid amounts seem to be rounded to the dollar, which is not expected!'
            )
        ], nullable=False),
        'REG': pa.Column(str, checks=[
            pa.Check(
                lambda s: 0.6 < s.value_counts(normalize=True).get('DUBAI', 0) < 0.9,
                name="dubai_proportion_range",
                error="Percentage of claims in 'DUBAI' is outside the 0.6-0.9 range."
            ),
            pa.Check(
                lambda s: 0.01 < s.value_counts(normalize=True).get('SHJ', 0) < 0.15,
                name="shj_proportion_range",
                error="Percentage of claims in 'SHJ' is outside the 0.01-0.15 range."
            ),
            pa.Check(
                lambda s: 0.01 < s.value_counts(normalize=True).get('AD', 0) < 0.1,
                name="ad_proportion_range",
                error="Percentage of claims in 'AD' is outside the 0.01-0.1 range."
            )
        ], nullable=True)
    },
    checks = [
        pa.Check(
            lambda df: df['VALUATION_DATE'] >= df['DATE_OF_ACCIDENT'],
            name='valuation date should be after accident date',
            error="Error: valuation date is before accident date"
        ),
        pa.Check(
            lambda df: latest_acc_month in df['DATE_OF_ACCIDENT'].dt.strftime('%Y-%m').unique(),
            name='ensure_latest_accident_month_present',
            error=f'The latest accident month {latest_acc_month} is missing!'
        ),
        pa.Check(
            lambda df: latest_val_month in df['VALUATION_DATE'].dt.strftime('%Y-%m').unique(),
            name='ensure_latest_valuation_month_present',
            error=f'The latest valuation month {latest_val_month} is missing!'
        ),
        pa.Check(
            lambda df: df.groupby(df['DATE_OF_ACCIDENT'].dt.to_period('M'))['PAID_AMOUNT'].sum().gt(0).all(),
            name='no_zero_loss_accident_months',
            error='There are accident months for which there are no losses!'
        ),

        # CHECK 4: Ensure every valuation month has losses > 0
        pa.Check(
            lambda df: df.groupby(df['VALUATION_DATE'].dt.to_period('M'))['PAID_AMOUNT'].sum().gt(0).all(),
            name='no_zero_loss_valuation_months',
            error='There are valuation months for which there are no losses!'
        ),
    ]
)

In [7]:
# we log errors encountered instead of failing immediately. These are saved in the specified location.
try:
    claims_schema.validate(claims, lazy=True)
    print('No errors found')
except pa.errors.SchemaErrors as e:
    print('There are some errors!')
    generate_dq_report(e, VALIDATION_REPORT_PATH)

There are some errors!
Validation report saved to: pandera_report.html
